In [0]:
SELECT 
    DATE_TRUNC('MONTH', SERVICE_DATE) AS month,
    COUNT(MEDICAL_EVENT_ID) AS claim_count
FROM com_raw.kom_medical_events
GROUP BY 1
ORDER BY 1;

In [0]:
 select * FROM com_edp_prd.com_raw.crx_patients

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient_territory AS
WITH base AS (
    SELECT
        d.crx_patient_id,
        z.region_id,
        z.region_name,
        z.territory_id,
        z.territory_name
    FROM com_edp_prd.com_raw.crx_dispenses d
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON d.shipment_hcp_zip = z.zipcode
    WHERE d.ndc IN ('1234-9999-04', '84976-0001-01')
      AND d.fill_type = 'Paid'
      AND d.returned_flag = 'N'
      AND d.crx_patient_id IS NOT NULL
)

SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    COUNT(DISTINCT crx_patient_id) AS distinct_patient_count
FROM base
WHERE territory_name IS NOT NULL
GROUP BY
    region_id,
    region_name,
    territory_id,
    territory_name

UNION ALL

SELECT
    NULL AS region_id,
    'All Region' AS region_name,
    NULL AS territory_id,
    'All Territory' AS territory_name,
    COUNT(DISTINCT crx_patient_id) AS distinct_patient_count
FROM base
WHERE territory_name IS NOT NULL;

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.patient_territory

In [0]:
SELECT * FROM com_intgr.sp_patients 

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.hub_patient_territory AS

WITH base AS (
    SELECT
        p.crx_patient_id,
        z.region_id,
        z.region_name,
        z.territory_id,
        z.territory_name
    FROM com_intgr.sp_patients p

    LEFT JOIN com_intgr.sp_hcp h
        ON p.current_crx_hcp_id = h.crx_hcp_id

    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON COALESCE(h.`hcp_address:_zip/postal_code`, h.hcp_address_zip_postal_code) = z.zipcode

    WHERE
        p.is_current = true
)

SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    COUNT(DISTINCT crx_patient_id) AS hub_pt_count
FROM base
GROUP BY
    region_id,
    region_name,
    territory_id,
    territory_name

UNION ALL

SELECT
    NULL AS region_id,
    'All Region' AS region_name,
    NULL AS territory_id,
    'All Territory' AS territory_name,
    COUNT(DISTINCT crx_patient_id) AS hub_pt_count
FROM base;

In [0]:
SELECT * FROM com_edp_prd.cmpa_insights_internal_schema.hub_patient_territory;

In [0]:
select * from com_intgr.distribution_sd_shipments;

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.hco_territory AS
WITH
hcp_master AS (
    SELECT DISTINCT
        attributes_npinum AS npi
    FROM com_intgr.customer_hcp
    WHERE type = 'HCP'
      AND is_active = true
      AND attributes_npinum IS NOT NULL
),

hco_base AS (
    SELECT
        z.region_name,
        z.territory_name,
        COUNT(DISTINCT s.crx_account_id) AS hco_with_hcp_prescribed_tivi
    FROM com_intgr.distribution_sd_shipments s
    LEFT JOIN com_intgr.distribution_accounts a
        ON s.crx_account_id = a.crx_account_id
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON a.account_facility_zip = z.zipcode
    -- WHERE z.territory_name IS NOT NULL
    GROUP BY z.region_name, z.territory_name
),

npi_acc_mapping AS (
    SELECT DISTINCT
        a.crx_account_id,
        a.invoicedate,
        b.account_npi AS npi
    FROM com_intgr.distribution_sd_shipments a
    LEFT JOIN com_intgr.distribution_accounts b
        ON a.crx_account_id = b.crx_account_id
),

tier1_hco AS (
    SELECT DISTINCT
        a.npi__v AS npi
    FROM com_intgr.customer a
    JOIN com_intgr.tsf b
        ON a.id = b.account__v
    WHERE a.object_type__v = 'OOT00000000V292'
      AND a.__END_AT IS NULL
      AND b.__END_AT IS NULL
      AND b.my_target__v = 'true'
),

tier1_active AS (
    SELECT COUNT(DISTINCT npi) AS tier1_hco_with_hcp_prescribed_tivi
    FROM npi_acc_mapping
    WHERE npi IN (SELECT npi FROM tier1_hco)
),

tier1_total AS (
    SELECT COUNT(DISTINCT npi) AS Total_Tier1_HCO
    FROM tier1_hco
),

current_month AS (
    SELECT COUNT(DISTINCT crx_account_id) AS current_month_tier1_hco
    FROM npi_acc_mapping
    WHERE npi IN (SELECT npi FROM tier1_hco)
      AND invoicedate >= DATE_TRUNC('month', CURRENT_DATE())
      AND invoicedate < DATE_TRUNC('month', ADD_MONTHS(CURRENT_DATE(), 1))
),

last_month AS (
    SELECT COUNT(DISTINCT crx_account_id) AS last_month_tier1_hco
    FROM npi_acc_mapping
    WHERE npi IN (SELECT npi FROM tier1_hco)
      AND invoicedate >= DATE_TRUNC('month', ADD_MONTHS(CURRENT_DATE(), -1))
      AND invoicedate < DATE_TRUNC('month', CURRENT_DATE())
),

hcp_tivi AS (
    SELECT DISTINCT npi
    FROM (
        SELECT PRESCRIBER_NPI AS npi FROM com_intgr.claims_pharmacy_events WHERE NDC11 = '69097022416'
        UNION
        SELECT REFERRING_NPI FROM com_intgr.claims_medical_events WHERE DIAGNOSIS_CODES = '|E761|' AND NDC11 = '69097022416'
        UNION
        SELECT RENDERING_NPI FROM com_intgr.claims_medical_events WHERE DIAGNOSIS_CODES = '|E761|' AND NDC11 = '69097022416'
        UNION
        SELECT BILLING_NPI FROM com_intgr.claims_medical_events WHERE DIAGNOSIS_CODES = '|E761|' AND NDC11 = '69097022416'
        UNION
        SELECT a.hcp_npi FROM com_intgr.sp_hcp a
    )
    WHERE npi IS NOT NULL
),

hcp_tivi_valid AS (
    SELECT DISTINCT t.npi
    FROM hcp_tivi t
    JOIN hcp_master h ON t.npi = h.npi
),

hcp_elaprase AS (
    SELECT DISTINCT npi
    FROM (
        SELECT PRESCRIBER_NPI AS npi FROM com_intgr.claims_pharmacy_events WHERE NDC11 = '54092070001'
        UNION
        SELECT REFERRING_NPI FROM com_intgr.claims_medical_events WHERE DIAGNOSIS_CODES = '|E761|' AND NDC11 = '54092070001'
        UNION
        SELECT RENDERING_NPI FROM com_intgr.claims_medical_events WHERE DIAGNOSIS_CODES = '|E761|' AND NDC11 = '54092070001'
        UNION
        SELECT BILLING_NPI FROM com_intgr.claims_medical_events WHERE DIAGNOSIS_CODES = '|E761|' AND NDC11 = '54092070001'
    )
    WHERE npi IS NOT NULL
),

hcp_elaprase_valid AS (
    SELECT DISTINCT e.npi
    FROM hcp_elaprase e
    JOIN hcp_master h ON e.npi = h.npi
),

tivi_hco AS (
    SELECT DISTINCT HCO_Primary_NPI
    FROM reltio_in_out.master_target_hcp_and_hco
    WHERE HCP_NPI IN (SELECT npi FROM hcp_tivi_valid)
      AND Is_Current = true
      AND end_date IS NULL
),

elaprase_hco AS (
    SELECT DISTINCT HCO_Primary_NPI
    FROM reltio_in_out.master_target_hcp_and_hco
    WHERE HCP_NPI IN (SELECT npi FROM hcp_elaprase_valid)
      AND Is_Current = true
      AND end_date IS NULL
),

tivi_sp AS (
    SELECT COUNT(DISTINCT HCO_Primary_NPI) AS tivi_hco_sp
    FROM reltio_in_out.master_target_hcp_and_hco
    WHERE HCP_NPI IN (
        SELECT DISTINCT b.hcp_npi
        FROM com_intgr.sp_dispense a
        LEFT JOIN com_intgr.sp_hcp b ON a.crx_hcp_id = b.crx_hcp_id
    )
),

tivi_hub AS (
    SELECT COUNT(DISTINCT HCO_Primary_NPI) AS tivi_hco_hub
    FROM reltio_in_out.master_target_hcp_and_hco
    WHERE HCP_NPI IN (
        SELECT DISTINCT b.hcp_npi
        FROM com_intgr.sp_patients a
        LEFT JOIN com_intgr.sp_hcp b ON a.current_crx_hcp_id = b.crx_hcp_id
    )
),

tivi_copay AS (
    SELECT COUNT(DISTINCT HCO_Primary_NPI) AS tivi_hco_copay
    FROM reltio_in_out.master_target_hcp_and_hco
    WHERE HCP_NPI IN (
        SELECT DISTINCT b.hcp_npi
        FROM com_intgr.sp_copay_transactions a
        LEFT JOIN com_intgr.sp_hcp b ON a.crx_hcp_id = b.crx_hcp_id
    )
),

territory_output AS (
    SELECT
        a.region_name,
        a.territory_name,
        a.hco_with_hcp_prescribed_tivi
    FROM hco_base a
)

SELECT
    region_name,
    territory_name,
    hco_with_hcp_prescribed_tivi
FROM territory_output

UNION ALL

SELECT
    'All Region',
    'All Territory',
    SUM(hco_with_hcp_prescribed_tivi)
FROM territory_output;

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.hco_territory

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.vials_territory AS
WITH
global_date_anchor AS (
    SELECT
        DATE_TRUNC('month', GREATEST(
            COALESCE((SELECT MAX(FILL_DATE) FROM com_intgr.claims_pharmacy_events WHERE NDC11 = '69097022416'), '1900-01-01'),
            COALESCE((SELECT MAX(SERVICE_DATE) FROM com_intgr.claims_medical_events WHERE NDC11 = '69097022416'), '1900-01-01'),
            COALESCE((SELECT MAX(ship_date) FROM com_intgr.sp_dispense), '1900-01-01')
        )) AS current_month_start
),

date_bounds AS (
    SELECT
        current_month_start,
        ADD_MONTHS(current_month_start, -1) AS previous_month_start
    FROM global_date_anchor
),

base_dispense AS (
    SELECT
        d.*,
        z.region_id,
        z.region_name,
        z.territory_id,
        z.territory_name
    FROM com_intgr.sp_dispense d
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON d.shipment_hcp_zip = z.zipcode
),

total_vials AS (
    SELECT
        region_id,
        region_name,
        territory_id,
        territory_name,
        SUM(quantity) AS total_vials
    FROM base_dispense
    WHERE territory_name IS NOT NULL
    GROUP BY
        region_id,
        region_name,
        territory_id,
        territory_name
),

sp_vials AS (
    SELECT
        region_id,
        region_name,
        territory_id,
        territory_name,
        SUM(quantity) AS sp_vials
    FROM base_dispense
    WHERE territory_name IS NOT NULL
      AND LOWER(pharmacy_name) LIKE '%orsini%'
    GROUP BY
        region_id,
        region_name,
        territory_id,
        territory_name
),

hub_vials AS (
    SELECT
        d.region_id,
        d.region_name,
        d.territory_id,
        d.territory_name,
        SUM(d.quantity) AS hub_vials
    FROM com_intgr.sp_patients a
    JOIN base_dispense d
        ON a.current_crx_hcp_id = d.crx_hcp_id
    WHERE d.territory_name IS NOT NULL
      AND a.latest_status_source = 'Occam'
    GROUP BY
        d.region_id,
        d.region_name,
        d.territory_id,
        d.territory_name
),

territory_output AS (
    SELECT
        t.region_id,
        t.region_name,
        t.territory_id,
        t.territory_name,
        t.total_vials,
        COALESCE(s.sp_vials, 0) AS sp_vials,
        COALESCE(h.hub_vials, 0) AS hub_vials,
        ROUND(TRY_DIVIDE(COALESCE(s.sp_vials, 0) * 100.0, t.total_vials), 2) AS sp_percentage,
        ROUND(TRY_DIVIDE(COALESCE(h.hub_vials, 0) * 100.0, t.total_vials), 2) AS hub_percentage
    FROM total_vials t
    LEFT JOIN sp_vials s
        ON t.territory_id = s.territory_id
    LEFT JOIN hub_vials h
        ON t.territory_id = h.territory_id
)

SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    total_vials,
    sp_vials,
    hub_vials,
    sp_percentage,
    hub_percentage
FROM territory_output

UNION ALL

SELECT
    NULL AS region_id,
    'All Region' AS region_name,
    NULL AS territory_id,
    'All Territory' AS territory_name,
    SUM(total_vials) AS total_vials,
    SUM(sp_vials) AS sp_vials,
    SUM(hub_vials) AS hub_vials,
    ROUND(TRY_DIVIDE(SUM(sp_vials) * 100.0, SUM(total_vials)), 2) AS sp_percentage,
    ROUND(TRY_DIVIDE(SUM(hub_vials) * 100.0, SUM(total_vials)), 2) AS hub_percentage
FROM territory_output;

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.vials_territory

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.hco_territory;

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.AL_PATIENT_OVERVIEW AS

WITH all_keys AS (
    SELECT region_id, region_name, territory_id, territory_name 
    FROM com_edp_prd.cmpa_insights_internal_schema.patient_territory

    UNION

    SELECT region_id, region_name, territory_id, territory_name 
    FROM com_edp_prd.cmpa_insights_internal_schema.hub_patient_territory

    UNION 

    SELECT region_id, region_name, territory_id, territory_name
    FROM com_edp_prd.cmpa_insights_internal_schema.hco_territory

    UNION

    SELECT region_id, region_name, territory_id, territory_name 
    FROM com_edp_prd.cmpa_insights_internal_schema.vials_territory
)

SELECT
    k.region_id,
    k.region_name,
    k.territory_id,
    k.territory_name,

    p.distinct_patient_count AS sp_pt_count,
    hub.hub_pt_count AS hub_pt_count,
    0 AS field_pt_count,

    h.hco_with_hcp_prescribed_tivi,

    v.total_vials,
    v.sp_vials,
    v.hub_vials,
    v.sp_percentage,
    v.hub_percentage

FROM all_keys k

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.patient_territory p
    ON k.territory_id = p.territory_id
    OR (k.territory_id IS NULL AND p.territory_id IS NULL)

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.hub_patient_territory hub
    ON k.territory_id = hub.territory_id
    OR (k.territory_id IS NULL AND hub.territory_id IS NULL)

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.hco_territory h
    ON k.territory_id = h.territory_id
    OR (k.territory_id IS NULL AND h.territory_id IS NULL)

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.vials_territory v
    ON k.territory_id = v.territory_id
    OR (k.territory_id IS NULL AND v.territory_id IS NULL);

In [0]:
 
-- insert into  cmpa_insights_internal_schema.reporting_kpi_metrics_master
-- SELECT
--     'Account_Leads' as dashboard_name,
--     'KPI_Overview' as Section_name,
--     region_id::string,
--     region_name,
--     territory_id::string,
--     territory_name,
--     kpi_name,
--     kpi_value
-- FROM AL_PATIENT_OVERVIEW
 
-- LATERAL VIEW STACK(9,
--     'sp_pt_count', NVL(CAST(sp_pt_count AS BIGINT),0),
--     'hub_pt_count', NVL(CAST(hub_pt_count AS BIGINT),0),
--     'field_pt_count', NVL(CAST(field_pt_count AS BIGINT),0),
--     'hco_with_hcp_prescribed_tivi', NVL(CAST(hco_with_hcp_prescribed_tivi AS BIGINT),0),
--     'total_vials', NVL(CAST(total_vials AS BIGINT),0),
--     'sp_vials', NVL(CAST(sp_vials AS BIGINT),0),
--     'hub_vials', NVL(CAST(hub_vials AS BIGINT),0),
--     'sp_percentage', NVL(CAST(sp_percentage AS BIGINT),0),
--     'hub_percentage', NVL(CAST(hub_percentage AS BIGINT),0)
-- ) AS kpi_name, kpi_value;
 
 

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.AL_PATIENT_OVERVIEW_BACKUP

In [0]:
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.AL_PATIENT_OVERVIEW AS
-- WITH
-- /*---------------------------------------------------------
-- 1. Prescriber master
-- ---------------------------------------------------------*/
-- prescriber_hcp AS (
--     SELECT DISTINCT
--         TRIM(attributes_npinum) AS npi
--     FROM com_intgr.customer_hcp
--     WHERE UPPER(type) = 'HCP'
-- ),

-- /*---------------------------------------------------------
-- 2. Individual providers
-- ---------------------------------------------------------*/
-- individual_providers AS (
--     SELECT DISTINCT
--         TRIM(npi) AS npi
--     FROM com_intgr.claims_providers
--     WHERE UPPER(provider_type) = 'INDIVIDUAL'
-- ),

-- /*---------------------------------------------------------
-- 3. Pharmacy prescribers
-- ---------------------------------------------------------*/
-- pharmacy_prescribers AS (
--     SELECT DISTINCT
--         TRIM(prescriber_npi) AS npi
--     FROM com_intgr.claims_pharmacy_events
--     WHERE ndc11 = '69097022416'
--       AND prescriber_npi IS NOT NULL
--       AND TRIM(prescriber_npi) <> ''
-- ),

-- /*---------------------------------------------------------
-- 4. Eligible medical claims
-- ---------------------------------------------------------*/
-- medical_claims AS (
--     SELECT
--         TRIM(referring_npi) AS ref_npi,
--         TRIM(rendering_npi) AS ren_npi,
--         TRIM(billing_npi) AS bill_npi
--     FROM com_intgr.claims_medical_events
--     WHERE diagnosis_codes = '|E761|'
--       AND (
--             TRIM(ndc11) = '69097022416'
--             OR TRIM(ndc11) = ''
--             OR ndc11 IS NULL
--           )
--       AND procedure_code IN ('J3490', 'J3590', 'J9999')
-- ),

-- /*---------------------------------------------------------
-- 5. Medical prescriber attribution
-- ---------------------------------------------------------*/
-- medical_prescribers AS (
--     SELECT DISTINCT
--         COALESCE(
--             CASE
--                 WHEN ref_npi IS NOT NULL
--                  AND ref_npi <> ''
--                  AND ref_npi IN (SELECT npi FROM prescriber_hcp)
--                 THEN ref_npi
--             END,
--             CASE
--                 WHEN ren_npi IS NOT NULL
--                  AND ren_npi <> ''
--                  AND ren_npi IN (SELECT npi FROM prescriber_hcp)
--                 THEN ren_npi
--             END,
--             CASE
--                 WHEN bill_npi IS NOT NULL
--                  AND bill_npi <> ''
--                  AND bill_npi IN (SELECT npi FROM prescriber_hcp)
--                  AND bill_npi IN (SELECT npi FROM individual_providers)
--                 THEN bill_npi
--             END
--         ) AS npi
--     FROM medical_claims
-- ),

-- /*---------------------------------------------------------
-- 6. Claritas distinct HCP list
-- ---------------------------------------------------------*/
-- claritas_hcp AS (
--     SELECT DISTINCT
--         a.hcp_npi AS npi,
--         b.date
--     FROM com_intgr.sp_hcp a
--     INNER JOIN (
--         SELECT DISTINCT
--             crx_hcp_id,
--             ship_date AS date
--         FROM com_intgr.sp_dispense

--         UNION

--         SELECT DISTINCT
--             crx_hcp_id,
--             transaction_date AS date
--         FROM com_intgr.sp_copay_transactions

--         UNION

--         SELECT DISTINCT
--             current_crx_hcp_id AS crx_hcp_id,
--             enrollment_date AS date
--         FROM com_intgr.sp_patients
--     ) b
--         ON a.crx_hcp_id = b.crx_hcp_id
--     WHERE b.date IS NOT NULL
-- ),

-- /*---------------------------------------------------------
-- 7. Final distinct HCP list
-- ---------------------------------------------------------*/
-- final_hcp_list AS (
--     SELECT DISTINCT npi
--     FROM (
--         SELECT npi FROM pharmacy_prescribers
--         UNION ALL
--         SELECT npi FROM medical_prescribers
--         UNION ALL
--         SELECT npi FROM claritas_hcp
--     ) final_hcp
--     WHERE npi IS NOT NULL
-- ),

-- /*---------------------------------------------------------
-- 8. HCP to territory mapping
-- ---------------------------------------------------------*/
-- hcp_territory AS (
--     SELECT DISTINCT
--         z.region_id,
--         z.region_name,
--         z.territory_id,
--         z.territory_name,
--         TRIM(h.hcp_npi) AS npi
--     FROM com_intgr.sp_hcp h
--     LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
--         ON COALESCE(h.`hcp_address:_zip/postal_code`, h.hcp_address_zip_postal_code) = z.zipcode
--     WHERE z.territory_id IS NOT NULL
--       AND h.hcp_npi IS NOT NULL
-- ),

-- /*---------------------------------------------------------
-- 9. Territory base
-- ---------------------------------------------------------*/
-- hcp_prescribed_territory_base AS (
--     SELECT
--         ht.region_id,
--         ht.region_name,
--         ht.territory_id,
--         ht.territory_name,
--         COUNT(DISTINCT f.npi) AS hcp_prescribed
--     FROM final_hcp_list f
--     JOIN hcp_territory ht
--         ON f.npi = ht.npi
--     GROUP BY
--         ht.region_id,
--         ht.region_name,
--         ht.territory_id,
--         ht.territory_name
-- ),

-- /*---------------------------------------------------------
-- 10. Territory + ALL
-- ---------------------------------------------------------*/
-- hcp_prescribed_territory AS (
--     SELECT
--         CAST(region_id AS STRING) AS region_id,
--         region_name,
--         CAST(territory_id AS STRING) AS territory_id,
--         territory_name,
--         hcp_prescribed
--     FROM hcp_prescribed_territory_base

--     UNION ALL

--     SELECT
--         'All Territories' AS region_id,
--         'ALL' AS region_name,
--         'All Territories' AS territory_id,
--         'ALL' AS territory_name,
--         SUM(hcp_prescribed) AS hcp_prescribed
--     FROM hcp_prescribed_territory_base
-- ),

-- /*---------------------------------------------------------
-- 11. Final output
-- ---------------------------------------------------------*/
-- final_output AS (
--     SELECT
--         CASE
--             WHEN a.region_name = 'ALL' AND a.territory_name = 'ALL'
--                 THEN 'All Territories'
--             ELSE CAST(a.region_id AS STRING)
--         END AS region_id,

--         a.region_name,

--         CASE
--             WHEN a.region_name = 'ALL' AND a.territory_name = 'ALL'
--                 THEN 'All Territories'
--             ELSE CAST(a.territory_id AS STRING)
--         END AS territory_id,

--         a.territory_name,
--         a.sp_pt_count,
--         a.hub_pt_count,
--         a.field_pt_count,
--         a.hco_with_hcp_prescribed_tivi,
--         a.total_vials,
--         a.sp_vials,
--         a.hub_vials,
--         a.sp_percentage,
--         a.hub_percentage,
--         COALESCE(hp.hcp_prescribed, 0) AS hcp_prescribed
--     FROM com_edp_prd.cmpa_insights_internal_schema.AL_PATIENT_OVERVIEW_BACKUP a
--     LEFT JOIN hcp_prescribed_territory hp
--         ON (
--             CAST(a.territory_id AS STRING) = hp.territory_id
--         )
--         OR (
--             a.territory_id IS NULL
--             AND a.region_name = 'ALL'
--             AND a.territory_name = 'ALL'
--             AND hp.territory_id = 'All Territories'
--         )
-- ),

-- /*---------------------------------------------------------
-- 12. Dummy Southwest row if missing
-- ---------------------------------------------------------*/
-- southwest_dummy AS (
--     SELECT
--         '202' AS region_id,
--         'West' AS region_name,
--         '2009' AS territory_id,
--         'Southwest' AS territory_name,
--         0 AS sp_pt_count,
--         0 AS hub_pt_count,
--         0 AS field_pt_count,
--         0 AS hco_with_hcp_prescribed_tivi,
--         0 AS total_vials,
--         0 AS sp_vials,
--         0 AS hub_vials,
--         0 AS sp_percentage,
--         0 AS hub_percentage,
--         0 AS hcp_prescribed
--     WHERE NOT EXISTS (
--         SELECT 1
--         FROM final_output
--         WHERE territory_id = '2009'
--            OR territory_name = 'Southwest'
--     )
-- )

-- SELECT *
-- FROM final_output

-- UNION ALL

-- SELECT *
-- FROM southwest_dummy;

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.AL_PATIENT_OVERVIEW AS
WITH
base_patients AS (
    SELECT DISTINCT patient_id
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
),

medical_ndc_npis AS (
    SELECT DISTINCT
        p.patient_id,
        'MEDICAL_NDC' AS source_type,
        TRIM(COALESCE(m.rendering_npi, m.referring_npi)) AS npi
    FROM base_patients p
    INNER JOIN com_edp_prd.com_raw.kom_medical_events m
        ON p.patient_id = m.patient_id
    WHERE TRIM(m.ndc11) = '8497600101'
      AND COALESCE(m.rendering_npi, m.referring_npi) IS NOT NULL
      AND TRIM(COALESCE(m.rendering_npi, m.referring_npi)) <> ''
),

pharmacy_ndc_npis AS (
    SELECT DISTINCT
        p.patient_id,
        'PHARMACY_NDC' AS source_type,
        TRIM(r.prescriber_npi) AS npi
    FROM base_patients p
    INNER JOIN com_edp_prd.com_raw.kom_pharmacy_events r
        ON p.patient_id = r.patient_id
    WHERE TRIM(r.ndc11) = '8497600101'
      AND UPPER(TRIM(r.transaction_result)) = 'PAID'
      AND r.prescriber_npi IS NOT NULL
      AND TRIM(r.prescriber_npi) <> ''
),

procedure_code_npis AS (
    SELECT DISTINCT
        p.patient_id,
        'PROCEDURE_CODE' AS source_type,
        TRIM(COALESCE(m.rendering_npi, m.referring_npi)) AS npi
    FROM base_patients p
    INNER JOIN com_edp_prd.com_raw.kom_medical_events m
        ON p.patient_id = m.patient_id
    WHERE TRIM(m.procedure_code) IN ('J3490', 'J3590', 'J9999')
      AND COALESCE(m.rendering_npi, m.referring_npi) IS NOT NULL
      AND TRIM(COALESCE(m.rendering_npi, m.referring_npi)) <> ''
      AND m.service_date >= DATE '2026-03-01'
),

all_patient_npis AS (
    SELECT * FROM medical_ndc_npis
    UNION ALL
    SELECT * FROM pharmacy_ndc_npis
    UNION ALL
    SELECT * FROM procedure_code_npis
),

hcp_territory AS (
    SELECT DISTINCT
        a.npi,
        CAST(ztm.region_id AS STRING) AS region_id,
        ztm.region_name,
        CAST(ztm.territory_id AS STRING) AS territory_id,
        ztm.territory_name
    FROM all_patient_npis a
    LEFT JOIN com_edp_prd.com_raw.kom_providers kp
        ON TRIM(a.npi) = TRIM(kp.npi)
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping ztm
        ON LEFT(TRIM(kp.provider_zip), 5) = LEFT(TRIM(ztm.zipcode), 5)
    WHERE a.npi IS NOT NULL
      AND ztm.territory_id IS NOT NULL
),

hcp_prescribed_territory_base AS (
    SELECT
        region_id,
        region_name,
        territory_id,
        territory_name,
        COUNT(DISTINCT npi) AS hcp_prescribed
    FROM hcp_territory
    GROUP BY
        region_id,
        region_name,
        territory_id,
        territory_name
),

territory_backbone AS (
    SELECT DISTINCT
        CAST(region_id AS STRING) AS region_id,
        region_name,
        CAST(territory_id AS STRING) AS territory_id,
        territory_name
    FROM com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping
    WHERE territory_id IS NOT NULL
),

all_territories AS (
    SELECT
        region_id,
        region_name,
        territory_id,
        territory_name
    FROM territory_backbone

    UNION ALL

    SELECT
        'All Territories' AS region_id,
        'ALL' AS region_name,
        'All Territories' AS territory_id,
        'ALL' AS territory_name
),

final_output AS (
    SELECT
        t.region_id,
        t.region_name,
        t.territory_id,
        t.territory_name,
        COALESCE(a.sp_pt_count, 0) AS sp_pt_count,
        COALESCE(a.hub_pt_count, 0) AS hub_pt_count,
        COALESCE(a.field_pt_count, 0) AS field_pt_count,
        COALESCE(a.hco_with_hcp_prescribed_tivi, 0) AS hco_with_hcp_prescribed_tivi,
        COALESCE(a.total_vials, 0) AS total_vials,
        COALESCE(a.sp_vials, 0) AS sp_vials,
        COALESCE(a.hub_vials, 0) AS hub_vials,
        COALESCE(a.sp_percentage, 0) AS sp_percentage,
        COALESCE(a.hub_percentage, 0) AS hub_percentage,
        COALESCE(h.hcp_prescribed, 0) AS hcp_prescribed
    FROM all_territories t
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.AL_PATIENT_OVERVIEW_BACKUP a
        ON t.territory_id = CASE
            WHEN a.region_name = 'ALL' AND a.territory_name = 'ALL'
                THEN 'All Territories'
            ELSE CAST(a.territory_id AS STRING)
        END
    LEFT JOIN (
        SELECT
            region_id,
            region_name,
            territory_id,
            territory_name,
            hcp_prescribed
        FROM hcp_prescribed_territory_base

        UNION ALL

        SELECT
            'All Territories' AS region_id,
            'ALL' AS region_name,
            'All Territories' AS territory_id,
            'ALL' AS territory_name,
            COUNT(DISTINCT npi) AS hcp_prescribed
        FROM hcp_territory
    ) h
        ON t.territory_id = h.territory_id
)

SELECT *
FROM final_output;

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.AL_PATIENT_OVERVIEW

In [0]:
-- TRUNCATE TABLE com_edp_prd.cmpa_insights_internal_schema.reporting_kpi_metrics_master;

In [0]:
INSERT INTO cmpa_insights_internal_schema.reporting_kpi_metrics_master
SELECT
    'Account_Leads' AS dashboard_name,
    'KPI_Overview' AS Section_name,
    region_id::string,
    region_name,
    territory_id::string,
    territory_name,
    kpi_name,
    kpi_value
FROM com_edp_prd.cmpa_insights_internal_schema.AL_PATIENT_OVERVIEW

LATERAL VIEW STACK(10,
    'sp_pt_count', NVL(CAST(sp_pt_count AS BIGINT),0),
    'hub_pt_count', NVL(CAST(hub_pt_count AS BIGINT),0),
    'field_pt_count', NVL(CAST(field_pt_count AS BIGINT),0),
    'hco_with_hcp_prescribed_tivi', NVL(CAST(hco_with_hcp_prescribed_tivi AS BIGINT),0),
    'total_vials', NVL(CAST(total_vials AS BIGINT),0),
    'sp_vials', NVL(CAST(sp_vials AS BIGINT),0),
    'hub_vials', NVL(CAST(hub_vials AS BIGINT),0),
    'sp_percentage', NVL(CAST(sp_percentage AS BIGINT),0),
    'hub_percentage', NVL(CAST(hub_percentage AS BIGINT),0),
    'hcp_prescribed', NVL(CAST(hcp_prescribed AS BIGINT),0)
) AS kpi_name, kpi_value;

In [0]:
SELECT DISTINCT
     s.crx_account_id , sum(quantity_shipped)
    FROM com_edp_prd.com_intgr.distribution_sd_shipments s
    inner JOIN (select distinct crx_account_id from  com_edp_prd.com_intgr.distribution_accounts ) a
        ON s.crx_account_id = a.crx_account_id
        group by 1

In [0]:
select * from com_edp_prd.com_intgr.distribution_sd_shipments